# 🔧 Configuración Centralizada - Arquitectura Medallion Churn Prediction

## Propósito
Este notebook contiene la configuración centralizada para el pipeline de churn prediction con arquitectura Medallion (Bronze → Silver → Gold).

## Contenido
1. **Catálogos y Esquemas** Unity Catalog
2. **Paths de Datos** Fuente y destino
3. **Diccionario de Tablas** Configuración de 5 fuentes de datos
4. **Funciones de Utilidad** Metadata de ingesta, validaciones
5. **Parámetros Delta Lake** Optimización y particionamiento
6. **Widgets de Orquestación** Para Databricks Workflows

## Uso
Este notebook debe ser importado mediante `%run` en notebooks Bronze, Silver y Gold:
```python
%run ./00_config
```

---
**Autor:** Data Engineering Team  
**Última actualización:** 2026-05-17  
**Versión:** 1.0

In [0]:
# Imports necesarios
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    current_timestamp, current_date, lit, col, 
    to_timestamp, to_date, trim, upper, lower,
    count, sum as _sum, avg, max as _max, min as _min,
    datediff, months_between, year, month, dayofmonth,
    when, coalesce, regexp_replace, split
)
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime, timedelta
import logging

# Configuración de logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Imports cargados correctamente")

In [0]:
# ================================================================================
# UNITY CATALOG CONFIGURATION
# ================================================================================

# Catálogo principal
CATALOG = "workspace"

# Esquemas por capa Medallion
BRONZE_SCHEMA = f"{CATALOG}.churn_bronze"
SILVER_SCHEMA = f"{CATALOG}.churn_silver"
GOLD_SCHEMA = f"{CATALOG}.churn_gold"

# Establecer catálogo actual
spark.sql(f"USE CATALOG {CATALOG}")

# Verificar que los esquemas existan
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    try:
        spark.sql(f"DESCRIBE SCHEMA {schema}")
        logger.info(f"✓ Schema verificado: {schema}")
    except Exception as e:
        logger.warning(f"⚠ Schema no encontrado: {schema}. Error: {str(e)}")
        # Crear el schema si no existe
        schema_name = schema.split('.')[-1]
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
        logger.info(f"✓ Schema creado: {schema}")

print("\n" + "="*80)
print("UNITY CATALOG CONFIGURATION")
print("="*80)
print(f"Catálogo:        {CATALOG}")
print(f"Bronze Schema:   {BRONZE_SCHEMA}")
print(f"Silver Schema:   {SILVER_SCHEMA}")
print(f"Gold Schema:     {GOLD_SCHEMA}")
print("="*80)

In [0]:
# ================================================================================
# DATA PATHS CONFIGURATION
# ================================================================================

# Path base del proyecto
BASE_PATH = "/Workspace/Users/rafael7cor7@gmail.com/BigData---Modelo-churn"

# Paths de datos fuente (CSV raw)
RAW_DATA_PATH = f"{BASE_PATH}/data/raw"

# Paths de checkpoint para streaming (si se usa Auto Loader)
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints"

# Paths de logs y auditoría
LOG_PATH = f"{BASE_PATH}/logs"

print("\n" + "="*80)
print("DATA PATHS CONFIGURATION")
print("="*80)
print(f"Base Path:       {BASE_PATH}")
print(f"Raw Data:        {RAW_DATA_PATH}")
print(f"Checkpoints:     {CHECKPOINT_PATH}")
print(f"Logs:            {LOG_PATH}")
print("="*80)

In [0]:
# ================================================================================
# SOURCE TABLES CONFIGURATION
# ================================================================================

# Configuración detallada de las 5 tablas fuente
SOURCE_TABLES_CONFIG = {
    "demo_asociados": {
        "file_name": "demo_asociados.csv",
        "file_path": f"{RAW_DATA_PATH}/demo_asociados.csv",
        "bronze_table": f"{BRONZE_SCHEMA}.raw_demo_asociados",
        "silver_table": f"{SILVER_SCHEMA}.asociados_demographics",
        "description": "Datos demográficos de asociados: sexo, fecha nacimiento, permanencia, nivel ingresos",
        "primary_key": "Nit",
        "partition_by": "load_date",
        "expected_columns": ["Nit", "Sexo", "Fechnacimiento", "Permanencia", 
                            "Antasociado", "Nivel_ingresos", "Participacion_social", "Uso_creditos"]
    },
    "retiro_aportes": {
        "file_name": "Retiro_aportes_2022.csv",
        "file_path": f"{RAW_DATA_PATH}/Retiro_aportes_2022.csv",
        "bronze_table": f"{BRONZE_SCHEMA}.raw_retiro_aportes",
        "silver_table": f"{SILVER_SCHEMA}.retiros_aportes",
        "description": "Retiros de aportes 2022: fechas, valores, motivos, tendencias",
        "primary_key": "Nit",
        "partition_by": "load_date",
        "expected_columns": ["Nit", "Fechasolicitud", "Fechaentrega", "Valoraportes",
                            "Motivo", "Oficina", "Castigo_cartera", "Tiempo_asociado",
                            "Frecuencia_retiros_previos", "Tendencia_saldo"]
    },
    "plano_ahorros": {
        "file_name": "Plano_ahorros.csv",
        "file_path": f"{RAW_DATA_PATH}/Plano_ahorros.csv",
        "bronze_table": f"{BRONZE_SCHEMA}.raw_plano_ahorros",
        "silver_table": f"{SILVER_SCHEMA}.productos_ahorros",
        "description": "Productos de ahorro: cantidad, saldos, variación, diversificación",
        "primary_key": "Nit",
        "partition_by": "load_date",
        "expected_columns": ["Nit", "Productos_ahorro", "Saldo_productos",
                            "Variacion_saldo_6m", "Diversificacion_productos"]
    },
    "detallado_operaciones": {
        "file_name": "detalladoperaciones.csv",
        "file_path": f"{RAW_DATA_PATH}/detalladoperaciones.csv",
        "bronze_table": f"{BRONZE_SCHEMA}.raw_detallado_operaciones",
        "silver_table": f"{SILVER_SCHEMA}.operaciones_transaccionales",
        "description": "Detalle de operaciones transaccionales: número, frecuencia, recencia",
        "primary_key": "Nit",
        "partition_by": "load_date",
        "expected_columns": ["Nit", "Numero_Tx", "Frecuencia_tx_mensual", "Ultima_tx_dias"]
    },
    "publiturno": {
        "file_name": "publiturno.csv",
        "file_path": f"{RAW_DATA_PATH}/publiturno.csv",
        "bronze_table": f"{BRONZE_SCHEMA}.raw_publiturno",
        "silver_table": f"{SILVER_SCHEMA}.interacciones_canal",
        "description": "Interacciones y canales: llegada agencia, línea, canal preferido, recencia",
        "primary_key": "Nit",
        "partition_by": "load_date",
        "expected_columns": ["Nit", "Llegada_agencia", "Interaccion_linea",
                            "Canal_preferido", "Tiempo_desde_ultima_interaccion"]
    }
}

# Tabla Gold final para churn prediction
GOLD_CHURN_TABLE = f"{GOLD_SCHEMA}.churn_prediction_features"

print("\n" + "="*80)
print("SOURCE TABLES CONFIGURATION")
print("="*80)
for table_name, config in SOURCE_TABLES_CONFIG.items():
    print(f"\n📊 {table_name}:")
    print(f"   File:         {config['file_name']}")
    print(f"   Bronze:       {config['bronze_table']}")
    print(f"   Silver:       {config['silver_table']}")
    print(f"   PK:           {config['primary_key']}")
    print(f"   Description:  {config['description']}")
print(f"\n🎯 Gold Table:    {GOLD_CHURN_TABLE}")
print("="*80)

In [0]:
# ================================================================================
# UTILITY FUNCTIONS - INGESTION METADATA
# ================================================================================

def add_ingestion_metadata(df: DataFrame, source_file: str) -> DataFrame:
    """
    Agrega columnas de metadata de ingesta a un DataFrame.
    
    Args:
        df: DataFrame de entrada
        source_file: Nombre del archivo fuente
    
    Returns:
        DataFrame con columnas de metadata agregadas:
        - timestamp_ingestion: Timestamp de cuando se ingirió el dato
        - source_file: Nombre del archivo origen
        - load_date: Fecha de carga (para particionamiento)
    
    Example:
        >>> df_with_metadata = add_ingestion_metadata(df, "demo_asociados.csv")
    """
    return df \
        .withColumn("timestamp_ingestion", current_timestamp()) \
        .withColumn("source_file", lit(source_file)) \
        .withColumn("load_date", current_date())


def validate_schema(df: DataFrame, expected_columns: list, table_name: str) -> bool:
    """
    Valida que el DataFrame contenga las columnas esperadas.
    
    Args:
        df: DataFrame a validar
        expected_columns: Lista de nombres de columnas esperadas
        table_name: Nombre de la tabla (para logging)
    
    Returns:
        True si todas las columnas esperadas están presentes
    
    Raises:
        ValueError: Si faltan columnas críticas
    """
    actual_columns = set(df.columns)
    expected_columns_set = set(expected_columns)
    
    missing_columns = expected_columns_set - actual_columns
    extra_columns = actual_columns - expected_columns_set
    
    if missing_columns:
        error_msg = f"❌ {table_name}: Faltan columnas {missing_columns}"
        logger.error(error_msg)
        raise ValueError(error_msg)
    
    if extra_columns:
        logger.warning(f"⚠ {table_name}: Columnas adicionales detectadas: {extra_columns}")
    
    logger.info(f"✓ {table_name}: Schema validado correctamente")
    return True


def log_ingestion_stats(df: DataFrame, table_name: str, operation: str = "LOAD") -> None:
    """
    Registra estadísticas de ingesta en logs.
    
    Args:
        df: DataFrame procesado
        table_name: Nombre de la tabla
        operation: Tipo de operación (LOAD, TRANSFORM, etc.)
    """
    row_count = df.count()
    col_count = len(df.columns)
    
    logger.info(f"""
    {'='*80}
    {operation} STATS - {table_name}
    {'='*80}
    Rows:       {row_count:,}
    Columns:    {col_count}
    Timestamp:  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
    {'='*80}
    """)
    
    print(f"✓ {operation} completado: {table_name} - {row_count:,} filas")


print("\n✓ Funciones de utilidad cargadas:")
print("  - add_ingestion_metadata()")
print("  - validate_schema()")
print("  - log_ingestion_stats()")

In [0]:
# ================================================================================
# DELTA LAKE OPTIMIZATION CONFIGURATION
# ================================================================================

# Configuración de OPTIMIZE y Z-ORDER
OPTIMIZE_CONFIG = {
    "enable_auto_optimize": True,
    "optimize_write": True,
    "auto_compact": True,
    "optimize_zorder_columns": {
        "bronze": ["Nit"],  # Solo Nit (load_date es columna de partición, no se puede usar en ZORDER)
        "silver": ["Nit"],
        "gold": ["Nit", "churn_score_percentile"]  # Se creará en Gold
    }
}

# Configuración de particionamiento
PARTITION_CONFIG = {
    "bronze_tables": "load_date",  # Particionar por fecha de carga
    "silver_tables": None,          # Sin particionamiento en Silver (tablas pequeñas)
    "gold_tables": None             # Sin particionamiento en Gold (tabla agregada)
}

# Configuración de retención de versiones Delta (días)
RETENTION_CONFIG = {
    "bronze": 30,   # 30 días de retención en Bronze
    "silver": 90,   # 90 días en Silver
    "gold": 180     # 180 días en Gold
}

# Configuración de compactación de archivos pequeños
COMPACTION_CONFIG = {
    "target_file_size_mb": 128,
    "auto_compact_enabled": True
}

def optimize_delta_table(table_name: str, zorder_columns: list = None) -> None:
    """
    Ejecuta OPTIMIZE en una tabla Delta con opcional Z-ORDER.
    
    Args:
        table_name: Nombre completo de la tabla (catalog.schema.table)
        zorder_columns: Lista de columnas para Z-ORDER (opcional)
    
    Example:
        >>> optimize_delta_table("workspace.churn_bronze.raw_demo_asociados", ["Nit"])
    """
    try:
        if zorder_columns:
            zorder_clause = ", ".join(zorder_columns)
            spark.sql(f"OPTIMIZE {table_name} ZORDER BY ({zorder_clause})")
            logger.info(f"✓ OPTIMIZE + ZORDER ejecutado en {table_name} por [{zorder_clause}]")
        else:
            spark.sql(f"OPTIMIZE {table_name}")
            logger.info(f"✓ OPTIMIZE ejecutado en {table_name}")
    except Exception as e:
        logger.error(f"❌ Error en OPTIMIZE de {table_name}: {str(e)}")
        raise


def vacuum_delta_table(table_name: str, retention_hours: int = 168) -> None:
    """
    Ejecuta VACUUM en una tabla Delta para limpiar archivos antiguos.
    
    Args:
        table_name: Nombre completo de la tabla
        retention_hours: Horas de retención (default: 168 = 7 días)
    
    Example:
        >>> vacuum_delta_table("workspace.churn_bronze.raw_demo_asociados", 720)  # 30 días
    """
    try:
        spark.sql(f"VACUUM {table_name} RETAIN {retention_hours} HOURS")
        logger.info(f"✓ VACUUM ejecutado en {table_name} (retención: {retention_hours}h)")
    except Exception as e:
        logger.error(f"❌ Error en VACUUM de {table_name}: {str(e)}")
        raise


print("\n" + "="*80)
print("DELTA LAKE OPTIMIZATION CONFIGURATION")
print("="*80)
print(f"Auto Optimize:        {OPTIMIZE_CONFIG['enable_auto_optimize']}")
print(f"Optimize Write:       {OPTIMIZE_CONFIG['optimize_write']}")
print(f"Auto Compact:         {OPTIMIZE_CONFIG['auto_compact']}")
print(f"Target File Size:     {COMPACTION_CONFIG['target_file_size_mb']} MB")
print(f"\nRetención de versiones:")
print(f"  Bronze:  {RETENTION_CONFIG['bronze']} días")
print(f"  Silver:  {RETENTION_CONFIG['silver']} días")
print(f"  Gold:    {RETENTION_CONFIG['gold']} días")
print(f"\nZ-ORDER Columns:")
print(f"  Bronze:  {OPTIMIZE_CONFIG['optimize_zorder_columns']['bronze']}")
print(f"  Silver:  {OPTIMIZE_CONFIG['optimize_zorder_columns']['silver']}")
print("="*80)

print("\n✓ Funciones de optimización cargadas:")
print("  - optimize_delta_table()")
print("  - vacuum_delta_table()")

In [0]:
# ================================================================================
# DATABRICKS WIDGETS - ORCHESTRATION PARAMETERS
# ================================================================================

# Crear widgets para parametrizar la ejecución desde Workflows
try:
    # Widget: Modo de ejecución (full o incremental)
    dbutils.widgets.dropdown(
        name="execution_mode",
        defaultValue="full",
        choices=["full", "incremental"],
        label="Modo de Ejecución"
    )
    
    # Widget: Fecha de procesamiento (para reprocesamiento histórico)
    dbutils.widgets.text(
        name="processing_date",
        defaultValue=datetime.now().strftime("%Y-%m-%d"),
        label="Fecha de Procesamiento"
    )
    
    # Widget: Nivel de logging
    dbutils.widgets.dropdown(
        name="log_level",
        defaultValue="INFO",
        choices=["DEBUG", "INFO", "WARNING", "ERROR"],
        label="Nivel de Logging"
    )
    
    # Widget: Ejecutar optimización Delta
    dbutils.widgets.dropdown(
        name="run_optimization",
        defaultValue="true",
        choices=["true", "false"],
        label="Ejecutar Optimización Delta"
    )
    
    # Widget: Capa a ejecutar (para testing)
    dbutils.widgets.dropdown(
        name="layer",
        defaultValue="all",
        choices=["all", "bronze", "silver", "gold"],
        label="Capa Medallion a Ejecutar"
    )
    
    print("✓ Widgets de orquestación creados")
    
except Exception as e:
    logger.warning(f"⚠ No se pudieron crear widgets (normal en entorno no-interactive): {str(e)}")

# Función para obtener parámetros de widgets
def get_execution_params() -> dict:
    """
    Obtiene los parámetros de ejecución desde los widgets de Databricks.
    
    Returns:
        Diccionario con parámetros de ejecución
    """
    try:
        params = {
            "execution_mode": dbutils.widgets.get("execution_mode"),
            "processing_date": dbutils.widgets.get("processing_date"),
            "log_level": dbutils.widgets.get("log_level"),
            "run_optimization": dbutils.widgets.get("run_optimization").lower() == "true",
            "layer": dbutils.widgets.get("layer")
        }
        logger.info(f"Parámetros de ejecución obtenidos: {params}")
        return params
    except:
        # Valores por defecto si no hay widgets
        return {
            "execution_mode": "full",
            "processing_date": datetime.now().strftime("%Y-%m-%d"),
            "log_level": "INFO",
            "run_optimization": True,
            "layer": "all"
        }

print("\n✓ Función de parámetros cargada: get_execution_params()")

In [0]:
# ================================================================================
# CONFIGURATION SUMMARY
# ================================================================================

print("\n" + "="*80)
print("🎯 CONFIGURACIÓN CENTRALIZADA CARGADA EXITOSAMENTE")
print("="*80)
print("\n📋 Componentes Disponibles:")
print("\n1. Unity Catalog:")
print(f"   • Catálogo: {CATALOG}")
print(f"   • Esquemas: {BRONZE_SCHEMA}, {SILVER_SCHEMA}, {GOLD_SCHEMA}")
print("\n2. Paths de Datos:")
print(f"   • Raw Data: {RAW_DATA_PATH}")
print(f"   • Checkpoints: {CHECKPOINT_PATH}")
print("\n3. Tablas Configuradas:")
print(f"   • Total de tablas fuente: {len(SOURCE_TABLES_CONFIG)}")
for table_name in SOURCE_TABLES_CONFIG.keys():
    print(f"     - {table_name}")
print("\n4. Funciones de Utilidad:")
print("   • add_ingestion_metadata()")
print("   • validate_schema()")
print("   • log_ingestion_stats()")
print("   • optimize_delta_table()")
print("   • vacuum_delta_table()")
print("   • get_execution_params()")
print("\n5. Configuración Delta Lake:")
print(f"   • Auto Optimize: {OPTIMIZE_CONFIG['enable_auto_optimize']}")
print(f"   • Target File Size: {COMPACTION_CONFIG['target_file_size_mb']} MB")
print("\n" + "="*80)
print("✅ Este notebook está listo para ser importado en notebooks Bronze/Silver/Gold")
print("   Uso: %run ./00_config")
print("="*80)